In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_at = dbutils.widgets.get("processed_at")
catalog = dbutils.widgets.get("catalog")

# these are required for populating the metadata column in the DataFrame, its also added to the job yml definition as base_parameters for the notebook task

In [ ]:
from utils.citibike_utils import get_trip_duration_mins, timestamp_to_date
from pyspark.sql.functions import create_map, lit


In [ ]:
df = spark.read.table(f"{catalog}.01_bronze.citibike")



In [17]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")


In [18]:
df = timestamp_to_date(df, "started_at", "start_date")



In [ ]:
df = df.withColumn(
    "metadata", 
    create_map(
        lit("pipeline_id"), lit(pipeline_id),
        lit("run_id"), lit(run_id),
        lit("task_id"), lit(task_id),
        lit("processed_at"), lit(processed_at)
    )
)

In [ ]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.02_silver.citibike")
)